<a href="https://colab.research.google.com/github/SridharS-Square/Agentic_AI_Workshop/blob/main/Building%20Advanced%20Al%20Agents%20with%20AutoGen/Financial_Portfolio_Manager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pyautogen google-generativeai

In [ ]:
# main_investment_advisor.py

import os
import google.generativeai as genai
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager

# --- 1. Configuration and Setup ---

def configure_api():
    """
    Sets up the API key for Google Generative AI.
    It's recommended to set the GOOGLE_API_KEY as an environment variable.
    """
    api_key = os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        api_key = input("Please enter your Google API Key: ").strip()
        print("API Key has been set for this session.")
    genai.configure(api_key=api_key)
    return api_key

API_KEY = configure_api()

# LLM configuration for the agents
GEMINI_LLM_CONFIG = {
    "config_list": [
        {
            "model": "gemini-1.5-flash",
            "api_key": API_KEY,
            "api_type": "google",
        }
    ],
    "temperature": 0.8,
}

# --- 2. Strategy Determination Logic ---

def get_investment_profile(financial_snapshot: dict) -> str:
    """
    Determines the investment profile (Growth or Value) based on financial data.

    Args:
        financial_snapshot: A dictionary containing salary and investment details.

    Returns:
        A string, either "GROWTH" or "VALUE".
    """
    total_assets = financial_snapshot.get('fixed_deposits', 0) + \
                   financial_snapshot.get('sips', 0) + \
                   financial_snapshot.get('real_estate', 0)

    annual_salary = financial_snapshot.get('salary', 0)

    # Simple heuristic: If total investments are less than 2x salary, focus on growth.
    if total_assets < annual_salary * 2:
        return "GROWTH"
    else:
        return "VALUE"

# --- 3. Agent Definitions ---

# Agent to represent the end-user
client_proxy = UserProxyAgent(
   name="Client_Proxy",
   human_input_mode="NEVER",
   system_message="You are the client. You initiate the request for financial advice by providing your portfolio and await the final, compiled report."
)

# Agent to perform initial analysis
analyst_agent = AssistantAgent(
    name="Financial_Analyst",
    llm_config=GEMINI_LLM_CONFIG,
    system_message="""You are a Financial Analyst. Your role is to perform an initial review of the client's portfolio.
    Your output should be a brief summary of their financial standing and a confirmation of the investment profile (Growth or Value) that has been decided."""
)

# Agent specializing in high-growth strategies
aggressive_investor_agent = AssistantAgent(
    name="Aggressive_Investment_Strategist",
    llm_config=GEMINI_LLM_CONFIG,
    system_message="""You are an Aggressive Investment Strategist. Your expertise is in high-risk, high-reward opportunities.
    Provide specific, actionable recommendations for growth-focused assets like equity funds, emerging tech stocks, and other suitable high-growth instruments. Justify your suggestions with potential upsides."""
)

# Agent specializing in stable, value-based strategies
conservative_investor_agent = AssistantAgent(
    name="Conservative_Investment_Strategist",
    llm_config=GEMINI_LLM_CONFIG,
    system_message="""You are a Conservative Investment Strategist. You specialize in capital preservation and steady, long-term returns.
    Provide specific, actionable recommendations for stable assets like debt instruments, blue-chip dividend stocks, and fixed-income securities. Emphasize the stability and reliability of your suggestions."""
)

# Agent to compile the final report
report_compiler_agent = AssistantAgent(
    name="Senior_Financial_Advisor",
    llm_config=GEMINI_LLM_CONFIG,
    system_message="""You are the Senior Financial Advisor. Your final task is to synthesize all the information provided by the Analyst and the Strategist.
    Combine the initial analysis with the specific investment recommendations to create a single, cohesive, and easy-to-read financial report for the client. The report should be well-structured and conclusive."""
)


# --- 4. Main Execution Workflow ---

def execute_investment_workflow():
    """
    Runs the end-to-end portfolio management process.
    """
    print("\n" + "="*50)
    print("🤖 Automated Investment Advisory System 🤖")
    print("="*50)

    # Step 1: Gather financial data from the user
    financial_snapshot = {
        'salary': float(input("Enter your current annual salary (e.g., 1200000): ")),
        'fixed_deposits': float(input("Enter your total Fixed Deposits (FD) amount: ")),
        'sips': float(input("Enter your total Systematic Investment Plans (SIPs) amount: ")),
        'real_estate': float(input("Enter the current market value of your real estate: "))
    }

    # Step 2: Determine the investment profile
    profile = get_investment_profile(financial_snapshot)
    print(f"\n✅ Profile Determined: Your profile suggests a **{profile}** investment strategy.")

    # Step 3: Dynamically select the expert agent based on the profile
    expert_agent = aggressive_investor_agent if profile == "GROWTH" else conservative_investor_agent
    print(f"✅ Expert Assigned: The **{expert_agent.name}** will provide recommendations.")

    # Step 4: Configure the group chat with the relevant agents
    advisory_team_chat = GroupChat(
        agents=[client_proxy, analyst_agent, expert_agent, report_compiler_agent],
        messages=[],
        max_round=10
    )

    manager = GroupChatManager(
        groupchat=advisory_team_chat,
        llm_config=GEMINI_LLM_CONFIG
    )

    # Step 5: Create the initial prompt to kick off the agent conversation
    chat_kickoff_prompt = f"""
    **New Client Financial Advisory Request**

    **Client's Financial Snapshot:**
    - Annual Salary: ₹{financial_snapshot['salary']:,.2f}
    - Fixed Deposits: ₹{financial_snapshot['fixed_deposits']:,.2f}
    - SIP Holdings: ₹{financial_snapshot['sips']:,.2f}
    - Real Estate Value: ₹{financial_snapshot['real_estate']:,.2f}

    **Pre-determined Strategy:** {profile}

    **Instructions for the Team:**
    1.  **Financial_Analyst**: Start by summarizing the client's financial position.
    2.  **{expert_agent.name}**: Based on the '{profile}' strategy, provide your specific investment suggestions.
    3.  **Senior_Financial_Advisor**: Conclude by compiling all analysis and recommendations into a final, unified report for the client.
    """

    # Step 6: Initiate the chat
    print("\n🚀 Initiating consultation with the AI advisory team... Please wait.\n")
    client_proxy.initiate_chat(
        recipient=manager,
        message=chat_kickoff_prompt,
    )

    print("\n" + "="*50)
    print("✅ Financial Advisory Process Complete!")
    total_portfolio = financial_snapshot['fixed_deposits'] + financial_snapshot['sips'] + financial_snapshot['real_estate']
    print(f"📊 Strategy Applied: {profile}")
    print(f"💼 Total Current Portfolio Value: ₹{total_portfolio:,.2f}")
    print("="*50)


if __name__ == "__main__":
    execute_investment_workflow()